# 返修患者随机效应模型与补充图
从随库患者对照复算各队列效应、REML 合并区间、逐队列剔除以及 S15–S17。打开本目录后顺序运行，结果写入 `03_result/06_公开复现` 或自选结果目录中的 `06_公开复现` 子目录。全计数提取和来源计数核对的单独运行方式见 README。

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
from importlib import import_module

import pandas as pd
from IPython.display import Image, display

# 在笔记本目录读取已归档的患者表，模型输出由下一步重新计算。
base = Path.cwd()
if not (base / "analysis/meta_patient_inputs.csv").is_file():
    raise FileNotFoundError("请在 revision_analysis 目录运行本笔记本")
sys.path.insert(0, str(base / "code"))
patient = import_module("00a_患者表达")
result_dir = patient.result_root() / "06_公开复现"
subprocess_env = {**os.environ, "PLOS_REVISION_DIR": str(result_dir.resolve())}
if (result_dir / "01_计数与预处理/09_队列细胞缓存").exists():
    raise ValueError("此目录已有全计数缓存，请用单独的结果目录运行公开复现")
model_outputs = {"meta_cohort_effects", "meta_summary", "meta_leave_one_out"}
for name, filename in patient.table_files.items():
    if name in model_outputs:
        continue
    target = result_dir / filename
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(base / "analysis" / (name + ".csv"), target)

input_table = pd.read_csv(result_dir / patient.table_files["meta_patient_inputs"])
print("患者对照记录", len(input_table), "条，模型", input_table["analysis"].nunique(), "种")

## 由患者效应拟合随机效应模型
每名患者在一个模型内提供一个表达差。先估计各队列的均值和均值方差，再计算带修正 Knapp–Hartung 区间的合并效应。

In [ ]:
# R 在本次结果目录重新拟合，用模型名选出两种基准注释的结果。
rscript = os.environ.get("RSCRIPT") or shutil.which("Rscript")
if not rscript:
    raise FileNotFoundError("请在含 metafor 的 R 环境运行，或设置 RSCRIPT")
fit = subprocess.run(
    [rscript, str(base / "code/04_拟合随机效应模型.R"), str(result_dir)], env=subprocess_env, capture_output=True, text=True
)
log = result_dir / "04_随机效应模型/05_R拟合日志.txt"
log.parent.mkdir(parents=True, exist_ok=True)
log.write_text(fit.stdout + fit.stderr, encoding="utf-8")
fit.check_returncode()
model_table = pd.read_csv(result_dir / patient.table_files["meta_summary"])
baseline_table = model_table.loc[model_table["analysis"].isin(["primary_labels", "uniform_labels"])]
fields = ["analysis", "k", "n", "estimate", "ci_low", "ci_high", "p_value", "tau2", "I2"]
display(baseline_table[fields + ["prediction_low", "prediction_high"]])

## 核对模型与随库参考表
SciPy 从队列均值和方差独立计算 REML。再按模型、队列与剔除对象对齐结果，逐列核对三张参考表。

In [ ]:
# 用同一 Python 环境独立核对。来源计数部分读取归档记录。
subprocess.run([sys.executable, str(base / "code/05a_核对患者与模型结果.py")], env=subprocess_env, check=True)
keys = {
    "meta_summary": ["analysis"], "meta_cohort_effects": ["analysis", "cohort"],
    "meta_leave_one_out": ["analysis", "omitted"],
}
for name, columns in keys.items():
    reference_table = pd.read_csv(base / "analysis" / (name + ".csv"))
    observed_table = pd.read_csv(result_dir / patient.table_files[name])
    reference_table = reference_table.set_index(columns, verify_integrity=True).sort_index()
    observed_table = observed_table.set_index(columns, verify_integrity=True).sort_index()
    pd.testing.assert_frame_equal(reference_table, observed_table, check_dtype=False, rtol=1e-6, atol=1e-8)
print("已核对模型", len(model_table), "种")

## 绘制补充图
S15 比较队列效应及合并值，S16 显示不同归一化与细胞筛选方法，S17 显示 GSE162631 各患者的细胞数、检出率及来源计数对照。

In [ ]:
subprocess.run([sys.executable, str(base / "code/05b_绘制返修补充图.py")], env=subprocess_env, check=True)
figure_dir = result_dir / "05_补充图与工作簿"
for name in ["02_S15随机效应森林图", "03_S16预处理比较", "04_S17来源计数核对"]:
    display(Image(filename=str(figure_dir / (name + ".png"))))

## 结果解读
以下统计从刚拟合的模型与来源计数汇总表读取。置信区间描述合并均值的不确定性，预测区间描述另一可比队列的可能效应。

In [ ]:
baseline_model_table = baseline_table.set_index("analysis", verify_integrity=True)
original = baseline_model_table.loc["primary_labels"]
uniform = baseline_model_table.loc["uniform_labels"]
prediction_crosses_zero = uniform.prediction_low <= 0 <= uniform.prediction_high
omission_table = pd.read_csv(result_dir / patient.table_files["meta_leave_one_out"])
uniform_omission_table = omission_table.loc[omission_table["analysis"] == "uniform_labels"]
crosses_zero = (uniform_omission_table["ci_low"] <= 0) & (uniform_omission_table["ci_high"] >= 0)
print(f"原注释的合并估计为 {original.estimate:.3f}。")
status = "包含零" if prediction_crosses_zero else "未包含零"
print(f"统一注释预测区间 [{uniform.prediction_low:.3f}, {uniform.prediction_high:.3f}]，{status}。")
print(f"逐队列剔除后，{int(crosses_zero.sum())}/{len(crosses_zero)} 个置信区间包含零。")